# DCE-MRI — source preparation and ROI generation

Case-wise preparation of three-channel DCE-MRI ROI crops from the study datasets, including patient-level partitioning and quality control.


## Étape 0 - Installation


In [ ]:

!pip -q install synapseclient nibabel SimpleITK opencv-python-headless scikit-image pandas tqdm pyarrow


## Étape 1 - Configuration


In [ ]:
from pathlib import Path
import os, re, json, shutil, zipfile, hashlib, math, gc, time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

SYNAPSE_ID = "syn60868042"
SYNAPSE_TOKEN = ""

USE_GOOGLE_DRIVE_OUTPUT = True  # recommandé : sauvegarde le dataset réduit dans Drive
MOUNT_DRIVE = True

RAW_CACHE_DIR = Path("/content/mama_mia_case_cache")
INVENTORY_DIR = Path("/content/mama_mia_inventory")

# ValueError: Mountpoint must not already contain files.
if USE_GOOGLE_DRIVE_OUTPUT:
    OUT_DIR = Path("/content/drive/MyDrive/ROI_MRI_Crops_256_v1")
else:
    OUT_DIR = Path("/content/ROI_MRI_Crops_256_v1")

NPZ_DIR = OUT_DIR / "npz"
LOG_DIR = OUT_DIR / "logs"
AUDIT_DIR = OUT_DIR / "audit"
FIG_DIR = OUT_DIR / "figures"

for p in [RAW_CACHE_DIR, INVENTORY_DIR]:
    p.mkdir(parents=True, exist_ok=True)

RUN_INVENTORY = True
RUN_PRECHECK = True
RUN_GENERATION = False
RUN_AUDIT = True
CREATE_AUDIT_LIGHT_ZIP = True
CREATE_FULL_ZIP_WITH_NPZ = False

MAX_CASES_PER_DATASET = {"ispy1": 2, "ispy2": 2, "nact": 2, "duke": 2}

IMAGE_SIZE = 256
ROI_MARGIN = 0.40
SLICE_AXIS = -1
NEGATIVES = False

PRE_PHASE_INDEX = 0
EARLY_POST_INDEX = 1
LATE_POST_INDEX = -1

P_LOW = 1.0
P_HIGH = 99.0

SPLIT_SEED = 42
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15
DEV_DATASETS = ["ispy1", "ispy2", "nact"]
EXTERNAL_DATASETS = ["duke"]

print("OUT_DIR:", OUT_DIR)
print("RUN_GENERATION:", RUN_GENERATION)
print("MAX_CASES_PER_DATASET:", MAX_CASES_PER_DATASET)



## Étape 2 - Monter Google Drive et vérifier l'espace


In [ ]:
import shutil, subprocess, os
from pathlib import Path

# Corrige: ValueError: Mountpoint must not already contain files

def _is_mounted(path="/content/drive"):
    return os.path.ismount(path)

if USE_GOOGLE_DRIVE_OUTPUT and MOUNT_DRIVE:
    from google.colab import drive
    mountpoint = Path("/content/drive")

    if mountpoint.exists() and (not _is_mounted(str(mountpoint))):
        existing = list(mountpoint.iterdir())
        if len(existing) > 0:
            print("/content/drive existe mais n'est pas monté et contient:", [p.name for p in existing[:10]])
            print("Suppression du faux point de montage local avant drive.mount...")
            shutil.rmtree(mountpoint)

    mountpoint.mkdir(parents=True, exist_ok=True)
    drive.mount(str(mountpoint), force_remount=False)

for p in [OUT_DIR, NPZ_DIR, LOG_DIR, AUDIT_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

for p in [RAW_CACHE_DIR, INVENTORY_DIR]:
    p.mkdir(parents=True, exist_ok=True)

usage = shutil.disk_usage("/content")
free_gb = usage.free / (1024**3)
total_gb = usage.total / (1024**3)
print(f"/content total: {total_gb:.1f} GB | free: {free_gb:.1f} GB")
!df -h /content

print("\nOUT_DIR:", OUT_DIR)
print("Note: le mode streaming ne doit jamais remplir /content avec les 88 Go complets.")



## Étape 3 - Connexion Synapse


In [ ]:

import synapseclient
from getpass import getpass

syn = synapseclient.Synapse()
if not SYNAPSE_TOKEN:
    SYNAPSE_TOKEN = getpass("Colle ton Synapse Personal Access Token puis Entrée: ")

syn.login(authToken=SYNAPSE_TOKEN, silent=True)
print("Connexion Synapse OK")


## Étape 4 - Inventaire Synapse sans téléchargement massif


In [ ]:

def safe_name(x):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(x))[:180]

def list_synapse_recursive(syn, root_id, max_depth=30):
    rows = []
    stack = [(root_id, "", 0)]
    visited = set()
    pbar = tqdm(desc="Synapse inventory", unit="entity")
    while stack:
        ent_id, rel_path, depth = stack.pop()
        if ent_id in visited or depth > max_depth:
            continue
        visited.add(ent_id)
        try:
            children = list(syn.getChildren(ent_id))
        except Exception as e:
            rows.append({"entity_id": ent_id, "path": rel_path, "name": "", "type": "ERROR", "error": str(e)})
            continue
        for child in children:
            cid = child.get("id")
            cname = child.get("name", "")
            ctype = child.get("type", "")
            cpath = f"{rel_path}/{cname}" if rel_path else cname
            rows.append({"entity_id": cid, "path": cpath, "name": cname, "type": ctype, "error": ""})
            if "Folder" in str(ctype) or str(ctype).endswith("Project"):
                stack.append((cid, cpath, depth + 1))
        pbar.update(1)
    pbar.close()
    return pd.DataFrame(rows)

inventory_csv = INVENTORY_DIR / "mama_mia_synapse_inventory.csv"
if RUN_INVENTORY or not inventory_csv.exists():
    inv = list_synapse_recursive(syn, SYNAPSE_ID)
    inv.to_csv(inventory_csv, index=False)
else:
    inv = pd.read_csv(inventory_csv)

print("Inventory rows:", len(inv))
display(inv.head(20))
print("Saved:", inventory_csv)


## Étape 5 - Classification des fichiers et appariement image/segmentation


In [ ]:


NIFTI_EXT_RE = re.compile(r"(\.nii|\.nii\.gz)$", re.I)
SEG_KEYWORDS = ["seg", "segmentation", "segmentations", "mask", "masks", "label", "labels", "tumor", "tumour", "expert", "corrected", "annotation"]
AUTO_BAD_KEYWORDS = ["automatic", "pred", "prediction", "preliminary"]

DATASET_ALIASES = {
    "ispy1": [r"ispy1", r"i[-_ ]?spy[-_ ]?1", r"ispy[-_ ]?1"],
    "ispy2": [r"ispy2", r"i[-_ ]?spy[-_ ]?2", r"ispy[-_ ]?2"],
    "nact":  [r"nact"],
    "duke":  [r"duke"],
}
STOP_TOKENS = {
    "mama", "mia", "mamamia", "mri", "breast", "cancer", "dce", "dynamic", "phase", "phases",
    "images", "image", "img", "mask", "masks", "seg", "segs", "segmentation", "segmentations",
    "label", "labels", "expert", "corrected", "annotation", "annotations", "nii", "gz", "nifti",
    "data", "dataset", "file", "files", "download", "raw", "original", "pre", "post", "early", "late",
    "t1", "t2", "axial", "sagittal", "coronal"
}

def _clean_text(s):
    return str(s).replace("\\", "/")

def infer_dataset(path):
    s = _clean_text(path).lower()
    for ds, pats in DATASET_ALIASES.items():
        for pat in pats:
            if re.search(rf"(^|[^a-z0-9]){pat}([^a-z0-9]|$)", s, flags=re.I):
                return ds
    return "unknown"

def _tokens(x):
    return [t for t in re.split(r"[^A-Za-z0-9]+", str(x)) if t]

def _valid_patient_token(t):
    tl = str(t).lower()
    if tl in STOP_TOKENS:
        return False
    if len(tl) < 2:
        return False
    if any(ch.isdigit() for ch in tl):
        return True
    if len(tl) >= 5 and tl not in STOP_TOKENS:
        return True
    return False

def infer_patient_token(path, dataset=None):
    """Infer patient token from Synapse path robustly.
    Works with patterns like:
      ISPY1_1234/file.nii.gz
      I-SPY1/1234/file.nii.gz
      MAMA-MIA/ISPY1/patient_1234/...
    """
    s = _clean_text(path)
    parts = [p for p in s.split("/") if p]
    ds = dataset or infer_dataset(s)
    alias_patterns = DATASET_ALIASES.get(ds, [])

    for i, part in enumerate(parts):
        low = part.lower()
        alias_hit = False
        rem = low
        for pat in alias_patterns:
            if re.search(pat, low, flags=re.I):
                alias_hit = True
                rem = re.sub(pat, " ", rem, flags=re.I)
        if alias_hit:
            for tok in _tokens(rem):
                if _valid_patient_token(tok):
                    return tok.upper()
            for part2 in parts[i+1:i+5]:
                for tok in _tokens(part2):
                    if _valid_patient_token(tok):
                        return tok.upper()

    if ds != "unknown":
        ds_pat = "|".join(alias_patterns)
        m = re.search(rf"(?:{ds_pat})[^A-Za-z0-9]+([A-Za-z0-9]{{2,}})", s, flags=re.I)
        if m and _valid_patient_token(m.group(1)):
            return m.group(1).upper()

    base = Path(s).name
    base = re.sub(r"\.nii(\.gz)?$", "", base, flags=re.I)
    for tok in _tokens(base):
        if _valid_patient_token(tok):
            return tok.upper()
    return None

def infer_case_id(path):
    ds = infer_dataset(path)
    tok = infer_patient_token(path, ds)
    if ds == "unknown" or tok is None:
        return None
    return f"{ds.upper()}_{tok}"

def infer_patient_id(case_id):
    if not case_id or "_" not in str(case_id):
        return None
    ds, pid = str(case_id).split("_", 1)
    return f"{ds}_{pid}"

def infer_phase_index(path):
    base = Path(str(path)).name
    for pat in [r"(?:_|-)(\d{4})(?:\.nii|\.nii\.gz)$", r"phase[_-]?(\d+)", r"ph[_-]?(\d+)", r"(?:^|[_-])t(\d+)(?:[_-]|\.)"]:
        m = re.search(pat, base, flags=re.I)
        if m:
            try:
                return int(m.group(1))
            except Exception:
                return None
    return None

def is_segmentation(path):
    s = str(path).lower()
    return any(k in s for k in SEG_KEYWORDS)

def seg_priority(path):
    s = str(path).lower()
    score = 0
    if "expert" in s: score -= 20
    if "corrected" in s: score -= 15
    if "segmentation" in s or "segmentations" in s: score -= 5
    if any(k in s for k in AUTO_BAD_KEYWORDS): score += 30
    return score

print("Inventory rows:", len(inv))
path_text = inv["path"].astype(str) + " / " + inv["name"].astype(str)
for key in ["ispy", "i-spy", "ispy1", "ispy2", "nact", "duke"]:
    n = int(path_text.str.contains(key, case=False, na=False, regex=False).sum())
    print(f"Inventory text contains '{key}':", n)

nii = inv[inv["name"].astype(str).str.contains(NIFTI_EXT_RE, regex=True, na=False)].copy()
nii["dataset"] = nii["path"].apply(infer_dataset)
nii["case_id"] = nii["path"].apply(infer_case_id)
nii["patient_id"] = nii["case_id"].apply(infer_patient_id)
nii["phase_index"] = nii["path"].apply(infer_phase_index)
nii["is_seg"] = nii["path"].apply(is_segmentation)
nii["seg_priority"] = nii["path"].apply(seg_priority)

print("NIfTI files:", len(nii))
print("Dataset counts among NIfTI:")
print(nii["dataset"].value_counts(dropna=False))
print("NIfTI by dataset/is_seg:")
print(nii.groupby(["dataset", "is_seg"]).size())

# Save examples for debugging if any dataset is missing
for ds in ["ispy1", "ispy2", "nact", "duke", "unknown"]:
    ex = nii[nii["dataset"] == ds][["entity_id", "path", "name", "case_id", "patient_id", "is_seg"]].head(20)
    if len(ex):
        ex.to_csv(LOG_DIR / f"debug_nifti_examples_{ds}.csv", index=False)
        print(f"Saved examples for {ds}:", LOG_DIR / f"debug_nifti_examples_{ds}.csv")

display(nii[["entity_id", "path", "name", "dataset", "case_id", "patient_id", "phase_index", "is_seg"]].head(30))
nii.to_csv(INVENTORY_DIR / "mama_mia_nifti_classified.csv", index=False)

image_rows = nii[(~nii["is_seg"]) & nii["case_id"].notna()].copy()
seg_rows = nii[(nii["is_seg"]) & nii["case_id"].notna()].copy()

cases = []
for case_id, g_img in image_rows.groupby("case_id"):
    g_seg = seg_rows[seg_rows["case_id"] == case_id].copy()
    dataset = g_img["dataset"].mode().iloc[0] if len(g_img) else "unknown"
    patient_id = infer_patient_id(case_id)
    phases = g_img.sort_values(["phase_index", "name"], na_position="last")
    if len(g_seg) > 0:
        seg = g_seg.sort_values(["seg_priority", "name"]).iloc[0]
        seg_entity_id = seg["entity_id"]
        seg_path = seg["path"]
    else:
        seg_entity_id = None
        seg_path = None
    cases.append({
        "case_id": case_id,
        "dataset": dataset,
        "patient_id": patient_id,
        "n_image_files": int(len(phases)),
        "phase_indices": ";".join(["NA" if pd.isna(x) else str(int(x)) for x in phases["phase_index"].tolist()]),
        "image_entity_ids": ";".join(phases["entity_id"].astype(str).tolist()),
        "image_paths": "||".join(phases["path"].astype(str).tolist()),
        "seg_entity_id": seg_entity_id,
        "seg_path": seg_path,
        "has_seg": seg_entity_id is not None,
    })

case_df = pd.DataFrame(cases)
if len(case_df):
    case_df = case_df.sort_values(["dataset", "case_id"]).reset_index(drop=True)
case_df.to_csv(INVENTORY_DIR / "mama_mia_case_pairs.csv", index=False)
print("Cases paired:", len(case_df))
if len(case_df):
    print("Case counts by dataset/has_seg:")
    print(case_df.groupby(["dataset", "has_seg"]).size())
    display(case_df.head(30))
else:
    print("No paired cases. Check debug_nifti_examples_unknown.csv and inventory paths.")


## Étape 6 - Correction ISPY1 et split patient-level


In [ ]:


valid_cases = case_df[(case_df["has_seg"] == True) & (case_df["dataset"].isin(DEV_DATASETS + EXTERNAL_DATASETS))].copy()
valid_cases["patient_id"] = valid_cases["case_id"].apply(infer_patient_id)

if len(valid_cases) == 0:
    raise RuntimeError(
        "No valid cases with both image and segmentation were paired. "
        "Open logs/debug_nifti_examples_*.csv and logs/mama_mia_case_pairs.csv to inspect naming."
    )

fix_report = valid_cases.groupby("dataset").agg(
    n_cases=("case_id", "nunique"),
    n_patients=("patient_id", "nunique"),
    min_images_per_case=("n_image_files", "min"),
    median_images_per_case=("n_image_files", "median"),
    max_images_per_case=("n_image_files", "max"),
).reset_index()
fix_report.to_csv(LOG_DIR / "ispy1_patient_id_fix_report.csv", index=False)
display(fix_report)

# Save examples for each dataset to verify patient derivation.
valid_cases[["dataset", "case_id", "patient_id", "n_image_files", "image_paths", "seg_path"]].head(100).to_csv(LOG_DIR / "patient_id_derivation_examples.csv", index=False)

ispy1_present_in_inventory = False
try:
    ispy1_present_in_inventory = int(nii["dataset"].eq("ispy1").sum()) > 0
except Exception:
    pass

ispy1_npat = int(fix_report.loc[fix_report["dataset"] == "ispy1", "n_patients"].iloc[0]) if (fix_report["dataset"] == "ispy1").any() else 0
ispy1_ncases = int(fix_report.loc[fix_report["dataset"] == "ispy1", "n_cases"].iloc[0]) if (fix_report["dataset"] == "ispy1").any() else 0

if ispy1_npat <= 3:
    # Do not bypass the ISPY1 patient-level correction before Track B generation.
    print("Dataset counts in NIfTI classification:")
    print(nii["dataset"].value_counts(dropna=False) if "nii" in globals() else "nii not available")
    print("Case counts in case_df:")
    print(case_df.groupby(["dataset", "has_seg"]).size() if len(case_df) else "case_df empty")
    raise RuntimeError(
        f"ISPY1 patient_id correction failed: {ispy1_npat} unique patients from {ispy1_ncases} paired ISPY1 cases. "
        "This usually means ISPY1 files were not discovered or the Synapse path naming was not parsed. "
        "Check logs/debug_nifti_examples_ispy1.csv, logs/debug_nifti_examples_unknown.csv, "
        "and /content/mama_mia_inventory/mama_mia_nifti_classified.csv. Do not continue until ISPY1 is plausible."
    )
print("OK: ISPY1 patient_id plausible:", ispy1_npat)

rng = np.random.default_rng(SPLIT_SEED)
dev_cases = valid_cases[valid_cases["dataset"].isin(DEV_DATASETS)].copy()
ext_cases = valid_cases[valid_cases["dataset"].isin(EXTERNAL_DATASETS)].copy()
patients = np.array(sorted(dev_cases["patient_id"].dropna().unique()))
if len(patients) == 0:
    raise RuntimeError("No development patients found after correction. Check dataset classification.")
rng.shuffle(patients)
n = len(patients)
n_train = int(round(n * TRAIN_FRAC))
n_val = int(round(n * VAL_FRAC))
train_p = set(patients[:n_train])
val_p = set(patients[n_train:n_train+n_val])
test_p = set(patients[n_train+n_val:])

def assign_split(row):
    if row["dataset"] in EXTERNAL_DATASETS:
        return "external"
    pid = row["patient_id"]
    if pid in train_p: return "train"
    if pid in val_p: return "validation"
    if pid in test_p: return "test"
    return "unknown"

valid_cases["split"] = valid_cases.apply(assign_split, axis=1)

sets = {s: set(valid_cases.loc[valid_cases["split"] == s, "patient_id"].dropna()) for s in ["train", "validation", "test", "external"]}
leaks = {
    "train_vs_validation": len(sets["train"] & sets["validation"]),
    "train_vs_test": len(sets["train"] & sets["test"]),
    "validation_vs_test": len(sets["validation"] & sets["test"]),
    "dev_vs_external": len((sets["train"] | sets["validation"] | sets["test"]) & sets["external"]),
}
print("Leakage:", leaks)
if any(v > 0 for v in leaks.values()):
    raise RuntimeError(f"Patient leakage detected: {leaks}")

valid_cases.to_csv(OUT_DIR / "case_level_split_manifest.csv", index=False)
summary = valid_cases.groupby(["dataset", "split"]).agg(n_cases=("case_id", "nunique"), n_patients=("patient_id", "nunique")).reset_index()
display(summary)
summary.to_csv(LOG_DIR / "case_level_split_summary.csv", index=False)
print("Saved:", OUT_DIR / "case_level_split_manifest.csv")


## Étape 7 - Fonctions de génération ROI DCE 3 canaux


In [ ]:

import nibabel as nib
import cv2
from scipy import ndimage as ndi

failures = []
manifest_rows = []

def clear_case_cache():
    if RAW_CACHE_DIR.exists():
        shutil.rmtree(RAW_CACHE_DIR, ignore_errors=True)
    RAW_CACHE_DIR.mkdir(parents=True, exist_ok=True)

def download_synapse_file(entity_id, case_cache):
    ent = syn.get(entity_id, downloadLocation=str(case_cache), ifcollision="overwrite.local")
    return Path(ent.path)

def load_nifti(path):
    img = nib.load(str(path))
    data = img.get_fdata(dtype=np.float32)
    return data, img

def select_three_phase_files(case_row):
    ids = str(case_row["image_entity_ids"]).split(";")
    paths = str(case_row["image_paths"]).split("||")
    rows = []
    for eid, p in zip(ids, paths):
        pi = infer_phase_index(p)
        rows.append({"entity_id": eid, "path": p, "phase_index": pi})
    dfp = pd.DataFrame(rows)
    if len(dfp) >= 2 and dfp["phase_index"].notna().any():
        dfp = dfp.sort_values("phase_index", na_position="last").reset_index(drop=True)
        pre = dfp.iloc[0]
        early = dfp.iloc[min(EARLY_POST_INDEX, len(dfp)-1)]
        late = dfp.iloc[-1]
        selected = [pre, early, late]
        fallback = len(set([x["entity_id"] for _, x in pd.DataFrame(selected).iterrows()])) < 3
        return [str(x["entity_id"]) for _, x in pd.DataFrame(selected).iterrows()], [x["phase_index"] for _, x in pd.DataFrame(selected).iterrows()], fallback
    return [str(ids[0])], [None], True

def normalize_3phase(vol3):
    vals = vol3[np.isfinite(vol3)]
    if vals.size == 0:
        return np.zeros_like(vol3, dtype=np.float32), {"p_low": 0, "p_high": 1}
    lo, hi = np.percentile(vals, [P_LOW, P_HIGH])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = float(np.nanmin(vals)), float(np.nanmax(vals))
        if hi <= lo:
            hi = lo + 1.0
    out = np.clip((vol3 - lo) / (hi - lo), 0, 1).astype(np.float32)
    return out, {"p_low": float(lo), "p_high": float(hi)}

def crop_square_with_margin(img2d_or_3ch, mask2d, margin=0.4, out_size=256):
    ys, xs = np.where(mask2d > 0)
    if len(xs) == 0:
        raise ValueError("empty_mask")
    x1, x2 = xs.min(), xs.max() + 1
    y1, y2 = ys.min(), ys.max() + 1
    bbox_w, bbox_h = x2 - x1, y2 - y1
    side = int(math.ceil(max(bbox_w, bbox_h) * (1.0 + 2.0 * margin)))
    side = max(side, 4)
    cx = (x1 + x2) / 2.0
    cy = (y1 + y2) / 2.0
    x0 = int(math.floor(cx - side / 2.0))
    y0 = int(math.floor(cy - side / 2.0))
    x3 = x0 + side
    y3 = y0 + side
    H, W = mask2d.shape
    crop_touches_border = x0 < 0 or y0 < 0 or x3 > W or y3 > H

    pad_left = max(0, -x0)
    pad_top = max(0, -y0)
    pad_right = max(0, x3 - W)
    pad_bottom = max(0, y3 - H)

    if img2d_or_3ch.ndim == 2:
        img_pad = np.pad(img2d_or_3ch, ((pad_top, pad_bottom), (pad_left, pad_right)), mode="constant", constant_values=0)
    else:
        img_pad = np.pad(img2d_or_3ch, ((pad_top, pad_bottom), (pad_left, pad_right), (0,0)), mode="constant", constant_values=0)
    mask_pad = np.pad(mask2d, ((pad_top, pad_bottom), (pad_left, pad_right)), mode="constant", constant_values=0)

    x0p, y0p = x0 + pad_left, y0 + pad_top
    x3p, y3p = x3 + pad_left, y3 + pad_top
    img_crop = img_pad[y0p:y3p, x0p:x3p]
    mask_crop = mask_pad[y0p:y3p, x0p:x3p]

    if img_crop.ndim == 2:
        img_res = cv2.resize(img_crop, (out_size, out_size), interpolation=cv2.INTER_LINEAR)
    else:
        chs = [cv2.resize(img_crop[..., c], (out_size, out_size), interpolation=cv2.INTER_LINEAR) for c in range(img_crop.shape[-1])]
        img_res = np.stack(chs, axis=-1)
    mask_res = cv2.resize(mask_crop.astype(np.uint8), (out_size, out_size), interpolation=cv2.INTER_NEAREST)
    mask_res = (mask_res > 0).astype(np.uint8)
    return img_res.astype(np.float32), mask_res, {
        "bbox_w": int(bbox_w), "bbox_h": int(bbox_h), "crop_size_native": int(side),
        "crop_touches_border": bool(crop_touches_border)
    }

def get_spacing_mm(nib_img, slice_axis=-1):
    zooms = list(nib_img.header.get_zooms())
    if len(zooms) < 3:
        return ""
    return ";".join([str(float(z)) for z in zooms[:3]])

def take_slice(vol, idx, axis=-1):
    return np.take(vol, idx, axis=axis)

print("Generation functions ready")


## Étape 8 - Pré-check sur quelques cas


In [ ]:

def limit_cases(df, max_cases_per_dataset):
    if max_cases_per_dataset is None:
        return df.copy()
    parts = []
    for ds, g in df.groupby("dataset"):
        m = max_cases_per_dataset.get(ds, 0)
        if m > 0:
            parts.append(g.head(m))
    return pd.concat(parts, ignore_index=True) if parts else df.head(0).copy()

run_cases = limit_cases(valid_cases, MAX_CASES_PER_DATASET)
print("Cases to process in this run:", len(run_cases))
display(run_cases[["case_id", "dataset", "split", "patient_id", "n_image_files", "seg_path"]].head(20))


## Étape 9 - Génération streaming des crops ROI 256 DCE


In [ ]:

def process_one_case(row):
    clear_case_cache()
    case_id = row["case_id"]
    dataset = row["dataset"]
    split = row["split"]
    patient_id = row["patient_id"]
    case_cache = RAW_CACHE_DIR / safe_name(case_id)
    case_cache.mkdir(parents=True, exist_ok=True)
    try:
        selected_ids, phase_indices, phase_fallback = select_three_phase_files(row)
        image_paths = []
        for eid in selected_ids:
            image_paths.append(download_synapse_file(eid, case_cache))
        seg_path = download_synapse_file(row["seg_entity_id"], case_cache)

        if len(image_paths) == 1:
            data, img_obj = load_nifti(image_paths[0])
            if data.ndim == 4 and data.shape[-1] >= 2:
                pre = data[..., PRE_PHASE_INDEX]
                early = data[..., min(EARLY_POST_INDEX, data.shape[-1]-1)]
                late = data[..., data.shape[-1]-1]
                vols = [pre, early, late]
                phase_indices = [PRE_PHASE_INDEX, min(EARLY_POST_INDEX, data.shape[-1]-1), data.shape[-1]-1]
            elif data.ndim == 3:
                vols = [data, data, data]
                phase_fallback = True
            else:
                raise ValueError(f"unsupported_image_shape_{data.shape}")
            ref_img = img_obj
        else:
            vols = []
            ref_img = None
            for p in image_paths:
                d, im = load_nifti(p)
                if d.ndim == 4:
                    d = d[..., 0]
                if d.ndim != 3:
                    raise ValueError(f"unsupported_phase_shape_{d.shape}")
                vols.append(d)
                ref_img = im if ref_img is None else ref_img

        mask, mask_obj = load_nifti(seg_path)
        if mask.ndim == 4:
            mask = mask[..., 0]
        mask = (mask > 0).astype(np.uint8)

        shapes = [v.shape for v in vols] + [mask.shape]
        if len(set(shapes)) != 1:
            raise ValueError(f"shape_mismatch_{shapes}")

        vol3 = np.stack(vols, axis=-1).astype(np.float32)
        vol3_norm, norm_info = normalize_3phase(vol3)

        axis = SLICE_AXIS if SLICE_AXIS >= 0 else mask.ndim + SLICE_AXIS
        lesion_slices = np.where(np.any(mask > 0, axis=tuple(i for i in range(mask.ndim) if i != axis)))[0]
        if len(lesion_slices) == 0:
            raise ValueError("empty_volume_mask")

        rows = []
        out_subdir = NPZ_DIR / dataset / split
        out_subdir.mkdir(parents=True, exist_ok=True)
        orig_ratio = float(mask.sum() / mask.size)
        spacing_mm = get_spacing_mm(ref_img, axis)

        for sl in lesion_slices:
            mask2d = take_slice(mask, int(sl), axis=axis)
            # Need 3 channel slice. vol3_norm dimensions: X,Y,Z,3; take slice from spatial axis, keep channel
            if axis == 0:
                img2d3 = vol3_norm[int(sl), :, :, :]
            elif axis == 1:
                img2d3 = vol3_norm[:, int(sl), :, :]
            else:
                img2d3 = vol3_norm[:, :, int(sl), :]

            if mask2d.shape != img2d3.shape[:2]:
                if mask2d.T.shape == img2d3.shape[:2]:
                    mask2d = mask2d.T
                else:
                    raise ValueError(f"slice_shape_mismatch_img{img2d3.shape}_mask{mask2d.shape}")
            if mask2d.sum() == 0 and not NEGATIVES:
                continue

            crop_img, crop_mask, crop_info = crop_square_with_margin(img2d3, mask2d, ROI_MARGIN, IMAGE_SIZE)
            if crop_mask.sum() == 0:
                raise ValueError("empty_crop_mask_after_resize")

            sample_id = f"mri_{case_id}_slice{int(sl):04d}"
            rel_npz = Path("npz") / dataset / split / f"{sample_id}.npz"
            out_npz = OUT_DIR / rel_npz
            np.savez_compressed(out_npz, image=crop_img.astype(np.float32), mask=crop_mask.astype(np.uint8))

            rows.append({
                "sample_id": sample_id,
                "dataset": dataset,
                "source": dataset,
                "split": split,
                "patient_id": patient_id,
                "case_id": case_id,
                "oracle_crop_flag": bool(split != "train"),
                "lesion_ratio_original": orig_ratio,
                "lesion_ratio_crop": float(crop_mask.sum() / crop_mask.size),
                "bbox_w": crop_info["bbox_w"],
                "bbox_h": crop_info["bbox_h"],
                "crop_size_native": crop_info["crop_size_native"],
                "crop_touches_border": crop_info["crop_touches_border"],
                "npz_path": str(rel_npz),
                "modality": "mri",
                "dce_phase": "pre;early;late",
                "dce_phase_indices": ";".join(["NA" if x is None or pd.isna(x) else str(int(x)) for x in phase_indices]),
                "dce_phase_fallback": bool(phase_fallback),
                "slice_index": int(sl),
                "volume_id": case_id,
                "plane": "native_slice_axis_last" if axis == 2 else f"native_axis_{axis}",
                "spacing_mm": spacing_mm,
                "norm_p_low": norm_info["p_low"],
                "norm_p_high": norm_info["p_high"],
            })
        return rows
    except Exception as e:
        failures.append({"case_id": case_id, "dataset": dataset, "split": split, "patient_id": patient_id, "reason": str(e)})
        return []
    finally:
        clear_case_cache()
        gc.collect()

if RUN_GENERATION:
    all_rows = []
    for _, row in tqdm(run_cases.iterrows(), total=len(run_cases), desc="Processing cases"):
        rows = process_one_case(row)
        all_rows.extend(rows)
        if all_rows:
            pd.DataFrame(all_rows).to_csv(OUT_DIR / "roi_mri_manifest_partial.csv", index=False)
        pd.DataFrame(failures).to_csv(OUT_DIR / "failures.csv", index=False)
    manifest = pd.DataFrame(all_rows)
    manifest.to_csv(OUT_DIR / "roi_mri_manifest.csv", index=False)
    pd.DataFrame(failures).to_csv(OUT_DIR / "failures.csv", index=False)
    print("Generated crops:", len(manifest))
    print("Failures:", len(failures))
    display(manifest.head())
else:
    print("RUN_GENERATION=False : génération non lancée.")


## Étape 10 - Audit GO/NO-GO


In [ ]:

def sha256_array(arr):
    h = hashlib.sha256()
    h.update(np.ascontiguousarray(arr).tobytes())
    return h.hexdigest()

def validate_npz(row):
    p = OUT_DIR / row["npz_path"]
    try:
        z = np.load(p)
        img = z["image"]
        mask = z["mask"]
        ok = True
        reasons = []
        if img.shape != (IMAGE_SIZE, IMAGE_SIZE, 3): ok=False; reasons.append(f"bad_image_shape_{img.shape}")
        if mask.shape != (IMAGE_SIZE, IMAGE_SIZE): ok=False; reasons.append(f"bad_mask_shape_{mask.shape}")
        if img.dtype != np.float32: ok=False; reasons.append(f"bad_image_dtype_{img.dtype}")
        if mask.dtype != np.uint8: ok=False; reasons.append(f"bad_mask_dtype_{mask.dtype}")
        if not (np.nanmin(img) >= -1e-6 and np.nanmax(img) <= 1+1e-6): ok=False; reasons.append("image_range_not_0_1")
        if not set(np.unique(mask)).issubset({0,1}): ok=False; reasons.append("mask_not_binary")
        if mask.sum() == 0: ok=False; reasons.append("empty_mask")
        return {"sample_id": row["sample_id"], "ok": ok, "reason": ";".join(reasons),
                "image_hash": sha256_array(img), "mask_hash": sha256_array(mask), "pair_hash": sha256_array(img) + sha256_array(mask)}
    except Exception as e:
        return {"sample_id": row.get("sample_id", "?"), "ok": False, "reason": str(e), "image_hash": "", "mask_hash": "", "pair_hash": ""}

report_lines = []
report_lines.append("# AUDIT_ROI_MRI_256_v1\n")
report_lines.append("## Mode\n")
report_lines.append(f"- RUN_GENERATION: `{RUN_GENERATION}`")
report_lines.append(f"- MAX_CASES_PER_DATASET: `{MAX_CASES_PER_DATASET}`")
report_lines.append(f"- OUT_DIR: `{OUT_DIR}`")
report_lines.append("\n## Case-level source inventory\n")
report_lines.append(f"- Valid paired cases: {len(valid_cases)}")
report_lines.append("\n")
report_lines.append(valid_cases.groupby(["dataset", "split"]).agg(n_cases=("case_id", "nunique"), n_patients=("patient_id", "nunique")).to_markdown())
report_lines.append("\n\n## Patient leakage\n")
for k,v in leaks.items():
    report_lines.append(f"- {k}: {v}")

if (OUT_DIR / "roi_mri_manifest.csv").exists():
    manifest = pd.read_csv(OUT_DIR / "roi_mri_manifest.csv")
    report_lines.append("\n## Crop counts\n")
    report_lines.append(manifest.groupby(["dataset", "split"]).agg(n_crops=("sample_id", "count"), n_patients=("patient_id", "nunique")).to_markdown())
    report_lines.append("\n\n## NPZ validation\n")
    val_rows = [validate_npz(row) for _, row in tqdm(manifest.iterrows(), total=len(manifest), desc="NPZ audit")]
    val_df = pd.DataFrame(val_rows)
    val_df.to_csv(AUDIT_DIR / "npz_full_validation.csv", index=False)
    report_lines.append(f"- NPZ checked: {len(val_df)}")
    report_lines.append(f"- NPZ OK: {int(val_df['ok'].sum())}/{len(val_df)}")
    report_lines.append(f"- unique image hashes: {val_df['image_hash'].nunique()}/{len(val_df)}")
    report_lines.append(f"- unique mask hashes: {val_df['mask_hash'].nunique()}/{len(val_df)}")
    report_lines.append(f"- unique pair hashes: {val_df['pair_hash'].nunique()}/{len(val_df)}")
    report_lines.append("\n## Lesion enrichment\n")
    report_lines.append(f"- median lesion_ratio_original: {manifest['lesion_ratio_original'].median():.6f}")
    report_lines.append(f"- median lesion_ratio_crop: {manifest['lesion_ratio_crop'].median():.6f}")
    enrich = manifest['lesion_ratio_crop'] / manifest['lesion_ratio_original'].replace(0, np.nan)
    report_lines.append(f"- median enrichment: {enrich.median():.2f}x")
    report_lines.append("\n## DCE fallback\n")
    if 'dce_phase_fallback' in manifest.columns:
        report_lines.append(str(manifest['dce_phase_fallback'].value_counts(dropna=False).to_dict()))
    if val_df['ok'].all() and all(v == 0 for v in leaks.values()):
        verdict = "GO" if MAX_CASES_PER_DATASET is None else "PRECHECK_GO_SUBSET_ONLY"
    else:
        verdict = "NO-GO"
else:
    report_lines.append("\n## Crop generation\n")
    report_lines.append("- No `roi_mri_manifest.csv` found because generation was not run.")
    verdict = "PRECHECK_ONLY"

if (OUT_DIR / "failures.csv").exists():
    fdf = pd.read_csv(OUT_DIR / "failures.csv")
    report_lines.append("\n## Failures\n")
    report_lines.append(f"- failure rows: {len(fdf)}")
    if len(fdf):
        report_lines.append(fdf['reason'].value_counts().head(20).to_markdown())
else:
    report_lines.append("\n## Failures\n- No failures.csv found yet.")

report_lines.append("\n## Verdict\n")
report_lines.append(f"**{verdict}**")

md_path = OUT_DIR / "AUDIT_ROI_MRI_256_v1.md"
with open(md_path, "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))
print("\n".join(report_lines))
print("Saved:", md_path)


## Étape 11 - ZIP léger d'audit et ZIP complet optionnel


In [ ]:

def zip_dir_selective(zip_path, root_dir, exclude_dirs=None):
    exclude_dirs = set(exclude_dirs or [])
    root_dir = Path(root_dir)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, allowZip64=True) as z:
        for p in root_dir.rglob("*"):
            if p.is_dir():
                continue
            rel = p.relative_to(root_dir)
            if any(part in exclude_dirs for part in rel.parts):
                continue
            z.write(p, arcname=str(Path(root_dir.name) / rel))
    return zip_path

if CREATE_AUDIT_LIGHT_ZIP:
    light_zip = Path("/content/ROI_MRI_Crops_256_v1_AUDIT_LIGHT.zip")
    zip_dir_selective(light_zip, OUT_DIR, exclude_dirs={"npz"})
    print("Audit light ZIP:", light_zip)

if CREATE_FULL_ZIP_WITH_NPZ:
    full_zip = Path("/content/ROI_MRI_Crops_256_v1_FULL_WITH_NPZ.zip")
    zip_dir_selective(full_zip, OUT_DIR, exclude_dirs=set())
    print("Full ZIP:", full_zip)
